In [26]:
def compute_mmd(x, y, sigma=1.0):
    def kernel_function(x, y, sigma=1.0):
        # 向量化计算核函数：x.shape=(m,d), y.shape=(n,d) -> 输出 (m,n)
        pairwise_dist = torch.cdist(x, y, p=2)  # 计算欧氏距离矩阵 (m,n)
        return torch.exp(-pairwise_dist ** 2 / (2 * sigma ** 2))
    m, n = x.size(0), y.size(0)
    
    # 计算核矩阵 (避免显式循环)
    K_xx = kernel_function(x, x, sigma)  # (m,m)
    K_yy = kernel_function(y, y, sigma)  # (n,n)
    K_xy = kernel_function(x, y, sigma)  # (m,n)
    
    # 计算 MMD (排除对角线元素)
    mmd = (K_xx.sum() - K_xx.diag().sum()) / (m * (m - 1)) + \
          (K_yy.sum() - K_yy.diag().sum()) / (n * (n - 1)) - \
          2 * K_xy.mean()
    return mmd

In [35]:
def compute_mmd_loop(x, y):
    def kernel_function(x, y):
        sigma = 1.0
        return torch.exp(-torch.norm(x - y) ** 2 / (2 * sigma ** 2))
    # Compute the MMD between two tensors x and y
    # x and y should have the same number of samples
    m = x.size(0)
    n = y.size(0)
    # Compute the kernel matrices for x and y
    xx_kernel = torch.zeros((m, m))
    yy_kernel = torch.zeros((n, n))
    xy_kernel = torch.zeros((m, n))
    # 修正1：计算上三角部分（不包括对角线）
    for i in range(m):
        for j in range(i + 1, m):  # j从i+1开始，跳过对角线
            xx_kernel[i, j] = kernel_function(x[i], x[j])
    xx_kernel = xx_kernel + xx_kernel.T  # 对称填充下三角

    for i in range(n):
        for j in range(i + 1, n):
            yy_kernel[i, j] = kernel_function(y[i], y[j])
    yy_kernel = yy_kernel + yy_kernel.T

    # 修正2：直接计算xy_kernel（无需对称性）
    for i in range(m):
        for j in range(n):
            xy_kernel[i, j] = kernel_function(x[i], y[j])

    # 对角线补1（因为之前未计算）
    xx_kernel.fill_diagonal_(1.0)
    yy_kernel.fill_diagonal_(1.0)

    mmd = (xx_kernel.sum() - m) / (m * (m - 1)) + (yy_kernel.sum() - n) / (n * (n - 1)) - 2 * xy_kernel.mean()
    return mmd

In [38]:
import torch
x = torch.randn(100, 10)
y = torch.randn(100, 10)*3

In [39]:
print (compute_mmd(x, y))
print (compute_mmd_loop(x, y))
print (torch.abs(compute_mmd(x, y) - compute_mmd_loop(x, y)).item())

tensor(0.0047)
tensor(0.0047)
9.313225746154785e-10
